# SunSight Stage 4 (GPU Colab/Kaggle): Training `YOLO11s-Seg` + Eksperimen CLAHE + Export ONNX

Notebook ini mengikuti spesifikasi [CLAUDE.md](../../CLAUDE.md) untuk melatih model segmentasi atap SunSight:
- **Arsitektur**: `yolo11s-seg.pt` (ringan di CPU ONNX, tajam menangkap tepi atap rapat Kota Bogor).
- **Resolusi**: `imgsz=1024` (atau `1280` bila VRAM GPU >= 16 GB, mis. T4/P100/L4) — jangan `640`!
- **Augmentasi anti-bias warna & orientasi nadir**: `hsv_h=0.04`, `hsv_s=0.7`, `hsv_v=0.5`, `degrees=180.0`, `flipud=0.5`, `fliplr=0.5`, `mosaic=1.0`, `close_mosaic=15`.
- **Eksperimen Jurnal SINTA 4 (Tabel 1)**: Opsi melatih versi **Raw RGB** vs **CLAHE LAB** pada dataset yang sama.
- **Output Akhir**: Bobot `best.onnx` (CPU-only, `simplify=True`) yang langsung bisa ditaruh ke `ml/service/weights/best.onnx`.

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime opencv-python-headless

In [ ]:
# 1. Ekstrak dataset ZIP hasil 04_split_dataset.py (sunsight_yolo11_seg.zip)
import zipfile
from pathlib import Path

DATASET_ZIP = Path('/content/sunsight_yolo11_seg.zip')
DATASET_DIR = Path('/content/sunsight_yolo11_seg')

if DATASET_ZIP.exists():
    with zipfile.ZipFile(DATASET_ZIP, 'r') as zf:
        zf.extractall(DATASET_DIR)
    print('Dataset diekstrak ke:', DATASET_DIR)
else:
    print('Upload sunsight_yolo11_seg.zip ke /content/ terlebih dahulu.')

In [ ]:
# 2. (Opsional untuk Tabel 1 Jurnal SINTA 4) Buat salinan dataset dengan pra-pemrosesan CLAHE LAB
import shutil
import cv2
from tqdm import tqdm

RUN_CLAHE_EXPERIMENT = False  # Ubah ke True untuk melatih versi CLAHE LAB

def apply_clahe_lab(bgr_img):
    lab = cv2.cvtColor(bgr_img, cv2.COLOR_BGR2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    merged = cv2.merge((clahe.apply(l), a, b))
    return cv2.cvtColor(merged, cv2.COLOR_LAB2BGR)

ACTIVE_DATASET = DATASET_DIR
if RUN_CLAHE_EXPERIMENT:
    CLAHE_DIR = Path('/content/sunsight_yolo11_seg_clahe')
    if CLAHE_DIR.exists():
        shutil.rmtree(CLAHE_DIR)
    shutil.copytree(DATASET_DIR, CLAHE_DIR)
    for split in ('train', 'val', 'test'):
        for img_p in tqdm(list((CLAHE_DIR / 'images' / split).glob('*.*')), desc=f'CLAHE {split}'):
            bgr = cv2.imread(str(img_p))
            if bgr is not None:
                cv2.imwrite(str(img_p), apply_clahe_lab(bgr), [int(cv2.IMWRITE_JPEG_QUALITY), 92])
    ACTIVE_DATASET = CLAHE_DIR
    print('Dataset CLAHE LAB siap di:', ACTIVE_DATASET)
else:
    print('Menggunakan dataset Esri RGB standar:', ACTIVE_DATASET)

In [ ]:
# 3. Training YOLO11s-Seg dengan augmentasi anti-bias warna & orientasi nadir
from ultralytics import YOLO

EXP_NAME = 'yolo11s_seg_esri_clahe' if RUN_CLAHE_EXPERIMENT else 'yolo11s_seg_esri_bogor'
IMGSZ = 1024  # Gunakan 1024 di T4 16GB (batch=4-8) atau 1280 di L4/A100

model = YOLO('yolo11s-seg.pt')
results = model.train(
    data=str(ACTIVE_DATASET / 'dataset.yaml'),
    epochs=120,
    patience=25,
    imgsz=IMGSZ,
    batch=4,
    optimizer='AdamW',
    lr0=0.001,
    cos_lr=True,
    # Jaga mask tiap atap tetap terpisah (krusial untuk permukiman padat Kota Bogor)
    overlap_mask=False,
    # Augmentasi anti-bias warna atap & rotasi nadir satelit
    hsv_h=0.04,
    hsv_s=0.7,
    hsv_v=0.5,
    degrees=180.0,
    flipud=0.5,
    fliplr=0.5,
    scale=0.35,
    mosaic=1.0,
    close_mosaic=15,
    project='/content/runs_sunsight',
    name=EXP_NAME,
    exist_ok=True,
    plots=True,
)

In [ ]:
# 4. Evaluasi resmi pada Test Set (catat mAP50, mAP50-95, Precision, Recall untuk Tabel 1 Jurnal)
best_pt = Path(f'/content/runs_sunsight/{EXP_NAME}/weights/best.pt')
best_model = YOLO(str(best_pt))

metrics = best_model.val(
    data=str(ACTIVE_DATASET / 'dataset.yaml'),
    split='test',
    imgsz=1280,
    conf=0.25,
    iou=0.50,
)
print('=== METRIK MASK TEST SET (JURNAL SINTA 4) ===')
print(f'Mask Precision : {metrics.seg.mp:.4f}')
print(f'Mask Recall    : {metrics.seg.mr:.4f}')
print(f'Mask mAP@50    : {metrics.seg.map50:.4f}')
print(f'Mask mAP@50-95 : {metrics.seg.map:.4f}')

In [ ]:
# 5. Ekspor ke ONNX (imgsz=1280, simplify=True) untuk FastAPI CPU-only SunSight
onnx_path = best_model.export(
    format='onnx',
    imgsz=1280,
    simplify=True,
    opset=17,
)
print('Model ONNX siap diunduh dan ditaruh ke ml/service/weights/best.onnx:', onnx_path)